# Etapa 13 · Verificación financiera

## Resumen
299 registros de 18 reportes: 290 cifras y 9 tipos de cambio. 94 conciliaciones y 64 comparaciones; tres diferencias de fuente documentadas, cuya causa sigue abierta.

## Contexto y método
Cuaderno complementario del HTML local. Ejecutar con el entorno Python del proyecto, sin red ni credenciales.

### Supuestos
Las tolerancias provienen de la precisión publicada. Los tipos de cambio son del propio reporte. Una conciliación contable no prueba causalidad ni disponibilidad histórica. No se cambia Gold. Fuentes: Bronze `aeromexico_ir`, Silver `aeromexico_ir_financial_history.parquet`, `docs/referencias/etapa-13/baseline.json` y `config/stage13_source_differences.json`.

In [1]:
from pathlib import Path
import os, sys, hashlib, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/analysis_agent/stage13.py').exists())
os.chdir(root)
sys.path.insert(0, str(root))
from src.analysis_agent.stage13 import build
from src.parse.aeromexico_ir_financial import run as extract
from src.config import PATHS

## Datos
Reextracción local y huellas para verificar idempotencia y conservación de Gold.

In [2]:
def hashes(folder):
    return {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(folder.glob('*.parquet'))}
gold_before = hashes(PATHS.gold)
extract()
first = hashlib.sha256((PATHS.silver / 'aeromexico_ir_financial_history.parquet').read_bytes()).hexdigest()
extract()
second = hashlib.sha256((PATHS.silver / 'aeromexico_ir_financial_history.parquet').read_bytes()).hexdigest()
assert first == second
assert gold_before == hashes(PATHS.gold)
print('Extracción idempotente; tablas Gold intactas.')

Extracción idempotente; tablas Gold intactas.


## Resultados
Conciliaciones y cobertura reconstruidas con los datos locales.

In [3]:
result = build()
assert len(result['rows']) == 299
assert len(result['checks']) == 94
assert len(result['comparison']) == 64
assert not result['unexplained']
conflicts = [r for r in result['comparison'] if r['status'] == 'documented_source_difference']
assert len(conflicts) == 3
print(json.dumps({'registros': len(result['rows']), 'faltantes_documentados':len(result['gaps']), 'conciliaciones':len(result['checks']), 'comparaciones':len(result['comparison']), 'diferencias_de_fuente':len(conflicts)}, indent=2))

{
  "registros": 299,
  "faltantes_documentados": 38,
  "conciliaciones": 94,
  "comparaciones": 64,
  "diferencias_de_fuente": 3
}


## Conclusiones
La extracción y las conversiones se pueden repetir. Los 38 campos objetivo no extraídos tienen disposición explícita: ausencia o definición distinta. Los cambios de fuente de 3T24 no se armonizan silenciosamente. La aceptación humana y la elegibilidad por fecha de corte permanecen pendientes; esta etapa no genera análisis de negocio.